In [5]:
%reload_ext autoreload
%autoreload 2

import importlib
import itertools
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd

import sim_instance_v3
importlib.reload(sim_instance_v3)

from model_modules.contract_v1 import ModuleSpec


# =============================================================================
# Overnight coarse sweep: smell-momentum oracle
# =============================================================================

OUT_DIR = Path("sweep_outputs")
OUT_DIR.mkdir(exist_ok=True)

RAW_PATH = OUT_DIR / "oracle_smell_decay_fill_coarse_50seed_raw.csv"
SUMMARY_PATH = OUT_DIR / "oracle_smell_decay_fill_coarse_50seed_summary.csv"


# -----------------------------------------------------------------------------
# Fixed run setup
# -----------------------------------------------------------------------------
SIM_LEN = 20_000
EVAL_LEN = 5_000

SEEDS = list(range(50))

env_kwargs = dict(
    radius=20,
    band=(9, 11),
    start_coord=(0, 0),
)

module_spec = ModuleSpec(
    orchestrator="oracle",
    pathfinder="oracle",
    eat="oracle",
    drink="oracle",
    explorer="smell_momentum",
)


# -----------------------------------------------------------------------------
# Coarse grid
# -----------------------------------------------------------------------------
DECAY_MULTS = [0.70, 0.80, 0.90]

# Broad around the previous best 1.60.
FILL_TARGETS = [1.30, 1.60, 1.90]

# Around previous useful momentum zone.
PERSIST_PS = [0.55, 0.75]

# Smell radius brackets current 12.
SMELL_RADII = [10, 14]

# Fixed smell explorer params.
TREND_EPS = 0.01
FOLLOW_P = 0.95
REVERSE_ON_DROP_P = 0.75
AVOID_REVERSE = True


configs = list(itertools.product(
    DECAY_MULTS,
    FILL_TARGETS,
    PERSIST_PS,
    SMELL_RADII,
))

print(f"configs: {len(configs)}")
print(f"seeds per config: {len(SEEDS)}")
print(f"total runs: {len(configs) * len(SEEDS)}")


# -----------------------------------------------------------------------------
# Resume support
# -----------------------------------------------------------------------------
if RAW_PATH.exists():
    df_existing = pd.read_csv(RAW_PATH)
    done_keys = set(
        zip(
            df_existing["seed"].astype(int),
            df_existing["decay_mult"].round(4),
            df_existing["fill_target"].round(4),
            df_existing["persist_p"].round(4),
            df_existing["smell_radius"].astype(int),
        )
    )
    rows = df_existing.to_dict("records")
    print(f"resuming: found {len(done_keys)} completed rows")
else:
    done_keys = set()
    rows = []


def make_oracle_params(fill_target, persist_p):
    oracle_params = sim_instance_v3.default_oracle_params()

    oracle_params["orchestrator"] = {
        "h_fill": fill_target,
        "s_fill": fill_target,
    }

    oracle_params["drink"] = {
        "fill_target": fill_target,
    }

    oracle_params["eat"] = {
        "fill_target": fill_target,
    }

    oracle_params["explorer"] = {
        "persist_p": persist_p,
        "avoid_reverse": AVOID_REVERSE,
        "trend_eps": TREND_EPS,
        "follow_p": FOLLOW_P,
        "reverse_on_drop_p": REVERSE_ON_DROP_P,
    }

    return oracle_params


def summarise_and_save(rows):
    df = pd.DataFrame(rows)
    df.to_csv(RAW_PATH, index=False)

    ok = df[df["error"].isna()].copy()

    summary = (
        ok.groupby(["decay_mult", "fill_target", "persist_p", "smell_radius"])
        .agg(
            mean_comfort_mean=("mean_comfort", "mean"),
            mean_comfort_std=("mean_comfort", "std"),

            eval_deaths_mean=("eval_deaths", "mean"),
            eval_deaths_median=("eval_deaths", "median"),
            eval_deaths_max=("eval_deaths", "max"),
            zero_death_seeds=("eval_deaths", lambda x: int((x == 0).sum())),

            eval_segments_mean=("eval_segments", "mean"),
            eval_segments_max=("eval_segments", "max"),
            eval_timeouts_mean=("eval_timeouts", "mean"),

            water_ticks_mean=("eval_water_ticks", "mean"),
            food_ticks_mean=("eval_food_ticks", "mean"),

            explorer_calls_mean=("explorer_calls", "mean"),
            smell_follow_mean=("smell_follow", "mean"),
            smell_reverse_mean=("smell_reverse", "mean"),
            momentum_persist_mean=("momentum_persist", "mean"),

            n=("seed", "count"),
        )
        .reset_index()
    )

    # Primary goal: deaths low.
    # Tie-breakers:
    #   - lower max deaths
    #   - more zero-death seeds
    #   - higher decay_mult preferred because it is the harder physics
    #   - higher comfort
    summary = summary.sort_values(
        [
            "eval_deaths_mean",
            "eval_deaths_max",
            "zero_death_seeds",
            "decay_mult",
            "mean_comfort_mean",
        ],
        ascending=[True, True, False, False, False],
    )

    summary.to_csv(SUMMARY_PATH, index=False)
    return df, summary


# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------
t0 = time.time()
run_i = len(done_keys)
total_runs = len(configs) * len(SEEDS)

for decay_mult, fill_target, persist_p, smell_radius in configs:
    for seed in SEEDS:
        key = (
            int(seed),
            round(float(decay_mult), 4),
            round(float(fill_target), 4),
            round(float(persist_p), 4),
            int(smell_radius),
        )

        if key in done_keys:
            continue

        run_i += 1

        oracle_params = make_oracle_params(
            fill_target=fill_target,
            persist_p=persist_p,
        )

        row = {
            "seed": seed,
            "decay_mult": decay_mult,
            "fill_target": fill_target,
            "persist_p": persist_p,
            "smell_radius": smell_radius,
            "avoid_reverse": AVOID_REVERSE,
            "trend_eps": TREND_EPS,
            "follow_p": FOLLOW_P,
            "reverse_on_drop_p": REVERSE_ON_DROP_P,
            "error": None,
        }

        try:
            # suppress per-run prints from sim_instance
            with contextlib.redirect_stdout(io.StringIO()):
                out = sim_instance_v3.sim_instance(
                    seed=seed,
                    sim_len=SIM_LEN,
                    eval_len=EVAL_LEN,

                    env_kwargs=env_kwargs,

                    module_spec=module_spec,
                    oracle_params=oracle_params,

                    curriculum_mode="band",
                    c_min=2,
                    c_max=9,
                    band_width=2,
                    curriculum_ramp_frac=0.6,
                    curriculum_sampling="uniform",
                    terminals_per_map=1,
                    life_cap=1000,

                    smell_radius=smell_radius,
                    decay_mult=decay_mult,

                    log_every=999_999,
                )

            stats = out.get("explorer_stats", {})

            row.update({
                "mean_comfort": out["mean_comfort"],
                "min_comfort": out["min_comfort"],
                "std_comfort": out["std_comfort"],

                "eval_deaths": out["death_count_eval"],
                "eval_death_rate": out["death_rate_eval"],
                "eval_timeouts": out["n_timeouts_eval"],
                "eval_segments": len(out["eval_segments"]),

                "eval_water_ticks": out["ticks_at_water_eval"],
                "eval_food_ticks": out["ticks_at_food_eval"],

                "train_wf": out["train_wf_trips"],
                "train_fw": out["train_fw_trips"],

                "explorer_type": out.get("explorer_type"),
                "explorer_calls": stats.get("n_calls"),
                "smell_follow": stats.get("n_smell_follow"),
                "smell_reverse": stats.get("n_smell_reverse"),
                "momentum_persist": stats.get("n_persist"),
                "random_turns": stats.get("n_random_turns"),
            })

            print(
                f"[{run_i:04d}/{total_runs}] "
                f"seed={seed:02d} decay={decay_mult:.2f} fill={fill_target:.2f} "
                f"p={persist_p:.2f} smellR={smell_radius:02d} | "
                f"comfort={row['mean_comfort']:.3f} deaths={row['eval_deaths']} "
                f"segments={row['eval_segments']}"
            )

        except Exception as e:
            row["error"] = repr(e)

            print(
                f"[{run_i:04d}/{total_runs}] FAILED "
                f"seed={seed:02d} decay={decay_mult:.2f} fill={fill_target:.2f} "
                f"p={persist_p:.2f} smellR={smell_radius:02d} | {repr(e)}"
            )

        rows.append(row)
        done_keys.add(key)

        # checkpoint every run
        if run_i % 1 == 0:
            df_tmp, summary_tmp = summarise_and_save(rows)

        # quick leaderboard every 25 runs
        if run_i % 25 == 0:
            elapsed = time.time() - t0
            print(f"\n--- checkpoint after {run_i} runs | elapsed {elapsed/60:.1f} min ---")
            display(summary_tmp.head(10))


df_raw, summary = summarise_and_save(rows)

print("\nDONE")
print("raw saved to:", RAW_PATH)
print("summary saved to:", SUMMARY_PATH)

display(summary.head(30))

configs: 36
seeds per config: 50
total runs: 1800
[0001/1800] seed=00 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.889 deaths=1 segments=6
[0002/1800] seed=01 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.896 deaths=0 segments=5
[0003/1800] seed=02 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.897 deaths=3 segments=8
[0004/1800] seed=03 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.924 deaths=0 segments=5
[0005/1800] seed=04 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.920 deaths=0 segments=5
[0006/1800] seed=05 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.928 deaths=2 segments=7
[0007/1800] seed=06 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.896 deaths=1 segments=6
[0008/1800] seed=07 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.891 deaths=0 segments=5
[0009/1800] seed=08 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.894 deaths=2 segments=7
[0010/1800] seed=09 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.856 deaths=1 segments=6
[0011/1800

,decay_mult,fill_target,persist_p,smell_radius,mean_comfort_mean,mean_comfort_std,eval_deaths_mean,eval_deaths_median,eval_deaths_max,zero_death_seeds,eval_segments_mean,eval_segments_max,eval_timeouts_mean,water_ticks_mean,food_ticks_mean,explorer_calls_mean,smell_follow_mean,smell_reverse_mean,momentum_persist_mean,n
0,0.7,1.3,0.55,10,0.90342,0.024044,1.2,1.0,5,9,6.2,10,4.0,1136.72,449.08,729.72,196.0,50.6,119.32,25


[0026/1800] seed=25 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.867 deaths=0 segments=5
[0027/1800] seed=26 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.891 deaths=2 segments=7
[0028/1800] seed=27 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.892 deaths=2 segments=7
[0029/1800] seed=28 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.853 deaths=4 segments=9
[0030/1800] seed=29 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.941 deaths=0 segments=5
[0031/1800] seed=30 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.884 deaths=2 segments=7
[0032/1800] seed=31 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.902 deaths=1 segments=6
[0033/1800] seed=32 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.884 deaths=5 segments=10
[0034/1800] seed=33 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.910 deaths=2 segments=7
[0035/1800] seed=34 decay=0.70 fill=1.30 p=0.55 smellR=10 | comfort=0.912 deaths=1 segments=6
[0036/1800] seed=35 decay=0.70 fill=1.30 p=0.55 smellR=10 |

,decay_mult,fill_target,persist_p,smell_radius,mean_comfort_mean,mean_comfort_std,eval_deaths_mean,eval_deaths_median,eval_deaths_max,zero_death_seeds,eval_segments_mean,eval_segments_max,eval_timeouts_mean,water_ticks_mean,food_ticks_mean,explorer_calls_mean,smell_follow_mean,smell_reverse_mean,momentum_persist_mean,n
0,0.7,1.3,0.55,10,0.89533,0.024623,1.54,1.0,5,13,6.54,10,4.0,1119.32,438.28,757.84,203.52,52.76,124.52,50


[0051/1800] seed=00 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.889 deaths=1 segments=6
[0052/1800] seed=01 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.896 deaths=0 segments=5
[0053/1800] seed=02 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.897 deaths=3 segments=8
[0054/1800] seed=03 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.907 deaths=2 segments=7
[0055/1800] seed=04 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.914 deaths=0 segments=5
[0056/1800] seed=05 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.928 deaths=2 segments=7
[0057/1800] seed=06 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.897 deaths=1 segments=6
[0058/1800] seed=07 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.891 deaths=0 segments=5
[0059/1800] seed=08 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.894 deaths=2 segments=7
[0060/1800] seed=09 decay=0.70 fill=1.30 p=0.55 smellR=14 | comfort=0.856 deaths=1 segments=6
[0061/1800] seed=10 decay=0.70 fill=1.30 p=0.55 smellR=14 | 

...

,decay_mult,fill_target,persist_p,smell_radius,mean_comfort_mean,mean_comfort_std,eval_deaths_mean,eval_deaths_median,eval_deaths_max,zero_death_seeds,eval_segments_mean,eval_segments_max,eval_timeouts_mean,water_ticks_mean,food_ticks_mean,explorer_calls_mean,smell_follow_mean,smell_reverse_mean,momentum_persist_mean,n
0,0.7,1.3,0.55,10,0.895330,0.024623,1.54,1.0,5,13,6.54,10,4.0,1119.32,438.28,757.84,203.52,52.76,124.52,50
1,0.7,1.3,0.55,14,0.887743,0.027077,1.56,1.5,6,14,6.56,11,4.0,1106.70,429.68,748.12,204.70,53.58,121.12,50
5,0.7,1.6,0.55,14,0.819504,0.029533,1.60,1.0,6,11,6.60,11,4.0,1035.94,420.30,761.72,204.66,54.14,125.34,50
8,0.7,1.9,0.55,10,0.737915,0.019273,1.70,1.0,7,12,6.70,12,4.0,1003.46,465.64,777.12,205.10,53.26,129.08,50
9,0.7,1.9,0.55,14,0.735902,0.021629,1.84,1.0,7,12,6.84,12,4.0,1004.34,466.06,764.58,205.72,54.74,125.88,50
4,0.7,1.6,0.55,10,0.820967,0.029753,1.90,2.0,6,10,6.90,11,4.0,1045.92,422.04,782.20,207.20,55.28,130.22,50
21,0.8,1.9,0.55,14,0.741932,0.016661,2.08,2.0,8,7,7.08,13,4.0,1038.70,489.32,738.50,201.22,52.88,117.96,50
20,0.8,1.9,0.55,10,0.740854,0.019170,2.10,2.0,6,8,7.10,11,4.0,1037.30,494.60,765.04,200.78,53.38,125.74,50
13,0.8,1.3,0.55,14,0.896755,0.023178,2.14,2.0,6,7,7.14,11,4.0,1155.24,463.72,742.90,209.38,54.36,116.24,50
17,0.8,1.6,0.55,14,0.831089,0.020934,2.16,2.0,6,8,7.16,11,4.0,1110.80,444.98,732.24,203.00,53.68,115.84,50


[1776/1800] seed=25 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.749 deaths=4 segments=9
[1777/1800] seed=26 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.725 deaths=5 segments=10
[1778/1800] seed=27 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.727 deaths=6 segments=11
[1779/1800] seed=28 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.754 deaths=1 segments=6
[1780/1800] seed=29 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.736 deaths=3 segments=8
[1781/1800] seed=30 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.735 deaths=2 segments=7
[1782/1800] seed=31 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.748 deaths=2 segments=7
[1783/1800] seed=32 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.724 deaths=4 segments=9
[1784/1800] seed=33 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.767 deaths=2 segments=7
[1785/1800] seed=34 decay=0.90 fill=1.90 p=0.75 smellR=14 | comfort=0.754 deaths=2 segments=7
[1786/1800] seed=35 decay=0.90 fill=1.90 p=0.75 smellR=14 

,decay_mult,fill_target,persist_p,smell_radius,mean_comfort_mean,mean_comfort_std,eval_deaths_mean,eval_deaths_median,eval_deaths_max,zero_death_seeds,eval_segments_mean,eval_segments_max,eval_timeouts_mean,water_ticks_mean,food_ticks_mean,explorer_calls_mean,smell_follow_mean,smell_reverse_mean,momentum_persist_mean,n
0,0.7,1.3,0.55,10,0.895330,0.024623,1.54,1.0,5,13,6.54,10,4.0,1119.32,438.28,757.84,203.52,52.76,124.52,50
1,0.7,1.3,0.55,14,0.887743,0.027077,1.56,1.5,6,14,6.56,11,4.0,1106.70,429.68,748.12,204.70,53.58,121.12,50
5,0.7,1.6,0.55,14,0.819504,0.029533,1.60,1.0,6,11,6.60,11,4.0,1035.94,420.30,761.72,204.66,54.14,125.34,50
8,0.7,1.9,0.55,10,0.737915,0.019273,1.70,1.0,7,12,6.70,12,4.0,1003.46,465.64,777.12,205.10,53.26,129.08,50
9,0.7,1.9,0.55,14,0.735902,0.021629,1.84,1.0,7,12,6.84,12,4.0,1004.34,466.06,764.58,205.72,54.74,125.88,50
4,0.7,1.6,0.55,10,0.820967,0.029753,1.90,2.0,6,10,6.90,11,4.0,1045.92,422.04,782.20,207.20,55.28,130.22,50
21,0.8,1.9,0.55,14,0.741932,0.016661,2.08,2.0,8,7,7.08,13,4.0,1038.70,489.32,738.50,201.22,52.88,117.96,50
20,0.8,1.9,0.55,10,0.740854,0.019170,2.10,2.0,6,8,7.10,11,4.0,1037.30,494.60,765.04,200.78,53.38,125.74,50
13,0.8,1.3,0.55,14,0.896755,0.023178,2.14,2.0,6,7,7.14,11,4.0,1155.24,463.72,742.90,209.38,54.36,116.24,50
17,0.8,1.6,0.55,14,0.831089,0.020934,2.16,2.0,6,8,7.16,11,4.0,1110.80,444.98,732.24,203.00,53.68,115.84,50



DONE
raw saved to: sweep_outputs\oracle_smell_decay_fill_coarse_50seed_raw.csv
summary saved to: sweep_outputs\oracle_smell_decay_fill_coarse_50seed_summary.csv


,decay_mult,fill_target,persist_p,smell_radius,mean_comfort_mean,mean_comfort_std,eval_deaths_mean,eval_deaths_median,eval_deaths_max,zero_death_seeds,eval_segments_mean,eval_segments_max,eval_timeouts_mean,water_ticks_mean,food_ticks_mean,explorer_calls_mean,smell_follow_mean,smell_reverse_mean,momentum_persist_mean,n
0,0.7,1.3,0.55,10,0.895330,0.024623,1.54,1.0,5,13,6.54,10,4.0,1119.32,438.28,757.84,203.52,52.76,124.52,50
1,0.7,1.3,0.55,14,0.887743,0.027077,1.56,1.5,6,14,6.56,11,4.0,1106.70,429.68,748.12,204.70,53.58,121.12,50
5,0.7,1.6,0.55,14,0.819504,0.029533,1.60,1.0,6,11,6.60,11,4.0,1035.94,420.30,761.72,204.66,54.14,125.34,50
8,0.7,1.9,0.55,10,0.737915,0.019273,1.70,1.0,7,12,6.70,12,4.0,1003.46,465.64,777.12,205.10,53.26,129.08,50
9,0.7,1.9,0.55,14,0.735902,0.021629,1.84,1.0,7,12,6.84,12,4.0,1004.34,466.06,764.58,205.72,54.74,125.88,50
4,0.7,1.6,0.55,10,0.820967,0.029753,1.90,2.0,6,10,6.90,11,4.0,1045.92,422.04,782.20,207.20,55.28,130.22,50
21,0.8,1.9,0.55,14,0.741932,0.016661,2.08,2.0,8,7,7.08,13,4.0,1038.70,489.32,738.50,201.22,52.88,117.96,50
20,0.8,1.9,0.55,10,0.740854,0.019170,2.10,2.0,6,8,7.10,11,4.0,1037.30,494.60,765.04,200.78,53.38,125.74,50
13,0.8,1.3,0.55,14,0.896755,0.023178,2.14,2.0,6,7,7.14,11,4.0,1155.24,463.72,742.90,209.38,54.36,116.24,50
17,0.8,1.6,0.55,14,0.831089,0.020934,2.16,2.0,6,8,7.16,11,4.0,1110.80,444.98,732.24,203.00,53.68,115.84,50


In [6]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import world_v1 as world


def starvation_time_water_perfect(
    seed=0,
    radius=20,
    decay_mult=0.2,
    h_fixed=1.60,
    s0=1.60,
    max_ticks=20_000,
    day_len=50,
):
    """
    Best-case starvation clock:
        hydration is clamped to h_fixed every tick
        no food is eaten
        satiation decays using your real world_v1 function
    """
    np.random.seed(seed)

    h_scale, s_scale = world.decay_scaling(radius)
    h_scale *= decay_mult
    s_scale *= decay_mult

    s = float(s0)

    for t in range(max_ticks):
        b = world.brightness(t, day_len=day_len, noise=True)

        # perfect water management assumption
        h = float(h_fixed)

        s = world.decay_satiation(s, h, b, s_scale)
        s = float(np.clip(s, 0.0, world.SMAX))

        if s <= world.DEATH_THRESH:
            return t + 1

    return max_ticks


for decay_mult in [1.0, 0.7, 0.5, 0.3, 0.25, 0.2]:
    times = [
        starvation_time_water_perfect(
            seed=seed,
            decay_mult=decay_mult,
            h_fixed=1.60,
            s0=1.60,
        )
        for seed in range(100)
    ]

    print(
        f"decay_mult={decay_mult:.2f} | "
        f"mean={np.mean(times):.1f} | "
        f"median={np.median(times):.1f} | "
        f"min={np.min(times)} | "
        f"max={np.max(times)}"
    )

decay_mult=1.00 | mean=635.1 | median=635.5 | min=608 | max=666
decay_mult=0.70 | mean=906.8 | median=907.5 | min=872 | max=942
decay_mult=0.50 | mean=1268.5 | median=1268.0 | min=1217 | max=1317
decay_mult=0.30 | mean=2113.7 | median=2112.0 | min=2053 | max=2197
decay_mult=0.25 | mean=2536.8 | median=2535.0 | min=2475 | max=2633
decay_mult=0.20 | mean=3169.9 | median=3170.0 | min=3096 | max=3278


In [1]:
import importlib
import inspect

mod = importlib.import_module("model_modules.oracle_modules.consumer_oracle_v1")

print("file:", mod.__file__)
print("\nNames containing Oracle / Consumer / Eat / Drink:")
for name in dir(mod):
    if any(s in name.lower() for s in ["oracle", "consumer", "eat", "drink"]):
        obj = getattr(mod, name)
        print(" ", name, type(obj))

file: c:\Users\Adarsh Arun\Downloads\homeostatic-agents-main\prototypes\06_feudal\model_modules\oracle_modules\consumer_oracle_v1.py

Names containing Oracle / Consumer / Eat / Drink:
  Consumer <class 'abc.ABCMeta'>
  ConsumerObs <class 'type'>
  OracleConsumer <class 'abc.ABCMeta'>
  make_drink <class 'function'>
  make_eat <class 'function'>


In [1]:
from sweep_fn_v5 import load_and_report

PROTOTYPE_NAME = "05_generalisation"
EXPERIMENT_NAME = PROTOTYPE_NAME + "__" + "self_imitation_v1"

res, manifest, runs_df, summary_df = load_and_report(EXPERIMENT_NAME, PROTOTYPE_NAME)

loaded experiment: 05_generalisation__self_imitation_v1
configs: 3   runs: 72


,config,solved,95% wilson,clean,comfort_med,deaths_med,zero_death,food%,water%
0,sil_frac_0,0% (0/24),[0%–14%],4%,0.55,145.5,0%,2.1,44.5
1,sil_frac_0p25,0% (0/24),[0%–14%],4%,0.55,124.5,0%,1.4,59.2
2,sil_frac_0p5,0% (0/24),[0%–14%],4%,0.46,162.0,0%,2.2,43.8



config reports:

CONFIG: sil_frac_0
Solved rate: 0.0% (0/24 seeds), 95% Wilson [0.0%, 13.8%].
Clean-solve (crossed the valley, no death cap): 4.2% (1/24) — the gap to solved is the survival cost of the crossing.
Comfort: median 0.555, mean 0.550, std 0.052, range 0.423 to 0.634.
Deaths: median 145.5, mean 147.9, range 105 to 188, zero-death rate 0.0%.
Resource occupancy: food 2.07%, water 44.49%, water:food ratio 2.09.
Camp shape: water-camp score 0.46, dominant-cell occupancy 3.54%.
Water→food route: success rate 0.001, success count median 2.0, path efficiency 0.611, perfect-ish trip rate 0.000.
Food→water route: success rate 0.057, success count median 3.0, path efficiency 0.679.
Two-way route floor: 0.001; total successful resource trips median 5.0.
Bug check: max non-neighbour jumps = 3.

CONFIG: sil_frac_0p25
Solved rate: 0.0% (0/24 seeds), 95% Wilson [0.0%, 13.8%].
Clean-solve (crossed the valley, no death cap): 4.2% (1/24) — the gap to solved is the survival cost of the crossi